# Reto 2 — Visión: identificación, ángulo y recuperación en COIL-100

Notebook de punta a punta y reproducible: **adquisición de datos → EDA → exploración y selección de modelo →
evaluación de la receta productiva congelada → análisis de errores → costo → exportación del artefacto**. Toda la lógica vive en `src/vision/`;
aquí solo se orquesta y se muestran resultados. Semilla global 42. Los experimentos quedan registrados en
MLflow (`sqlite:///mlflow.db`, experimento `vision-coil100`).

**Regla metodológica:** todo hiperparámetro y todo umbral se elige con particiones de *validación* construidas
exclusivamente con las vistas de entrenamiento de cada protocolo; ningún número de test se usa para elegir nada.
Hay dos niveles: (i) **exploración** (sección 4), que compara clasificadores y reporta también su test a título
informativo; (ii) la **receta productiva congelada** (sección 5), definida una sola vez en `src/vision/recipe.py`,
que se ajusta en el train de cada protocolo y se evalúa **una única vez** en su test. Las cifras de cabecera son
las de la receta.

In [1]:
import sys, json, time, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from src.vision import data as vdata, analysis as va, experiments as vexp
from src.vision.angle import circular_error
from src.vision.data import ensure_dataset, sha256_file, data_version_hash, protocol_split, validation_split, COIL_SHA256
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
np.random.seed(42)
T0 = time.time()

## 1. Adquisición y verificación de datos

Si `data/raw/coil-100/` no existe, se descarga el zip oficial
(`https://www.cs.columbia.edu/CAVE/databases/SLAM_coil-20_coil-100/coil-100/coil-100.zip`), se verifica su
SHA-256 y se extrae. Además se calcula un **hash de versión de datos** (SHA-256 sobre nombre + contenido de
cada PNG) que viaja en MLflow y en `metadata.json` del artefacto.

In [2]:
img_dir = ensure_dataset()
zip_path = img_dir.parent / "coil-100.zip"
if zip_path.exists():
    print("SHA-256 zip:", sha256_file(zip_path), "OK" if sha256_file(zip_path) == COIL_SHA256 else "DISTINTO")
DATA_HASH = data_version_hash()
print("imágenes:", len(list(img_dir.glob("obj*__*.png"))), "| hash de versión de datos:", DATA_HASH[:16], "…")
data = vexp.load_all()          # imágenes + características clásicas + píxeles crudos + embeddings ResNet18 (cache en data/runtime)
ds = data["ds"]
print(ds.images.shape, "clases:", len(np.unique(ds.labels)), "ángulos:", np.unique(ds.angles)[:4], "…", np.unique(ds.angles)[-1])

SHA-256 zip: fcb4b8b01c0778f416d8d7a09f45082c4f7075cc74743dcddd5c44307bd4afe7 OK


imágenes: 7200 | hash de versión de datos: e9e73124fbf0e964 …


(7200, 128, 128, 3) clases: 100 ángulos: [ 0  5 10 15] … 355


## 2. Análisis exploratorio (EDA)

In [3]:
fig = va.sample_grid(ds); va.savefig(fig, "01_sample_grid"); plt.show()
st = va.image_stats(ds)
desc = pd.DataFrame({k: pd.Series(v).describe(percentiles=[.001, .01, .5, .99, .999]) for k, v in st.items()}).T
display(desc.round(3))

,count,mean,std,min,0.1%,1%,50%,99%,99.9%,max
brightness,7200.0,0.268,0.071,0.129,0.134,0.151,0.265,0.440,0.491,0.526
object_fraction,7200.0,0.471,0.147,0.128,0.131,0.185,0.468,0.781,0.930,0.952
background_median,7200.0,26.297,2.154,22.000,23.000,24.000,26.000,35.000,43.000,47.000
sharpness,7200.0,327.561,251.640,44.245,46.601,54.948,255.630,1282.385,1810.964,2041.270


**Lectura.** 100 objetos × 72 vistas (paso exacto de 5°), 128×128 RGB, **fondo oscuro uniforme** (mediana del
borde ≈ 25/255, *no* 0) e **iluminación controlada**: brillo y nitidez varían poco. Consecuencias de diseño:

* El problema de clase es "fácil" en laboratorio; el riesgo real está en el ángulo y en la extrapolación a planta.
* Como el fondo es casi constante, una máscara por umbral de gris (> 40) separa objeto/fondo; en planta esto no
  vale (cinta, reflejos) → compuerta de calidad + segmentación robusta serían requisitos.
* Las estadísticas de brillo/nitidez/fracción de objeto definen los umbrales de la **compuerta de calidad**
  determinista (`src/vision/quality.py`, percentil extremo × margen).

In [4]:
from src.vision.models import VisionModel
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
Zall = PCA(150, random_state=42).fit_transform(StandardScaler().fit_transform(data["classic"]))  # solo para EDA
sym = va.symmetry_analysis(Zall, ds)
fig = va.symmetry_figure(sym); va.savefig(fig, "02_symmetry"); plt.show()
symdf = pd.DataFrame({"perfil": sym["profile_score"], "loo": sym["loo_score"], "d180/d90": sym["ratio_180_90"]})
symdf.index = [f"obj{o}" for o in symdf.index]
print("Más rotacionalmente ambiguos según el perfil (d@5° / d@≥45°):")
display(symdf.sort_values("perfil", ascending=False).head(10).round(3))
print("Vista 'gemela' más parecida a ≥30° (índice LOO):")
display(symdf.sort_values("loo", ascending=False).head(8).round(3))
print(f"Objetos con vistas opuestas (180°) más parecidas que a 90°: {(symdf['d180/d90'] < 1).sum()} de 100")

Más rotacionalmente ambiguos según el perfil (d@5° / d@≥45°):


,perfil,loo,d180/d90
obj95,0.768,0.097,1.069
obj94,0.593,0.042,1.061
obj47,0.591,0.028,1.094
obj25,0.552,0.000,0.977
obj50,0.528,0.000,0.850
obj34,0.493,0.000,1.068
obj7,0.484,0.000,1.120
obj86,0.484,0.000,0.997
obj30,0.484,0.000,0.767
obj56,0.468,0.000,1.142


Vista 'gemela' más parecida a ≥30° (índice LOO):


,perfil,loo,d180/d90
obj67,0.358,0.472,0.276
obj98,0.365,0.403,0.294
obj84,0.351,0.347,0.294
obj12,0.321,0.236,0.311
obj1,0.383,0.194,0.416
obj51,0.275,0.153,0.321
obj65,0.292,0.111,0.482
obj96,0.387,0.111,0.504


Objetos con vistas opuestas (180°) más parecidas que a 90°: 75 de 100


**Simetría rotacional (cuantificada).** Para cada objeto se mide la distancia media en el espacio de
características entre pares de vistas en función de su separación angular. En un objeto asimétrico la curva
crece con la separación; en uno simétrico (latas, tazas, donas, frascos: p. ej. obj95, obj94, obj47, obj25)
es casi plana → **el ángulo es intrínsecamente ambiguo**. El índice LOO (¿la vista más parecida está a ≥30°?)
detecta además objetos con vistas a 180° casi idénticas (cajas/frascos con dos caras parecidas).

**Tratamiento:** (i) a nivel de instancia, `estimate_angle` marca `ambiguous` si existe una vista de referencia
a ≥30° del ángulo estimado con distancia ≤ 1.10 × la mejor; (ii) a nivel de clase, el modelo productivo
(referencias cada 5°) marca como simétricas las clases con índice LOO ≥ 0.25; (iii) **empíricamente**: en
cada protocolo, las clases cuyo ángulo falla (> ±10°) en más del 25 % de sus vistas de *validación* se marcan
como de ángulo poco fiable (detecta las donas/tazas obj47, obj94, obj95 sin mirar el test); (iv) todo esto genera
`rotational_ambiguity` → **REVIEW** (el robot recibe HOLD).

**Vistas a 180°:** en ~3/4 de los objetos la vista opuesta (180°) está más cerca en el espacio de
características que la vista a 90° (la silueta de un objeto visto desde lados opuestos es casi la imagen espejo).
Por eso el ángulo no se estima con una media circular de todos los vecinos (se cancelarían modos opuestos), sino
con el modo del vecino más cercano refinado localmente, y la ambigüedad se marca explícitamente. Alternativa documentada: para un objeto
perfectamente simétrico el ángulo no importa para el agarre y podría reportarse "módulo simetría"; no lo
hacemos por defecto porque la etiqueta/código de barras sí puede depender del ángulo.

In [5]:
pairs = va.confusable_pairs(Zall, ds, top=10)
display(pd.DataFrame(pairs).round(2))

,a,b,min_inter_distance,mean_intra_nn_distance
0,obj61,obj92,10.99,7.45
1,obj29,obj32,15.77,9.11
2,obj47,obj94,17.94,5.06
3,obj67,obj84,19.00,14.58
4,obj15,obj91,19.25,13.88
5,obj1,obj15,19.81,13.35
6,obj62,obj93,19.81,9.74
7,obj15,obj69,20.23,14.47
8,obj15,obj76,20.71,14.01
9,obj21,obj44,20.84,16.15


**Clases casi duplicadas.** Los pares con menor distancia inter-objeto (comparada con la distancia típica entre
vistas adyacentes del mismo objeto) son los candidatos a confusión: mismas formas con distinto color/texto
(p. ej. cajas o latas de la misma familia). Esto motiva incluir **color** (HSV) además de forma (HOG).

**Laboratorio vs. planta.** COIL-100 tiene fondo negro, luz controlada, cámara fija a la misma distancia,
un único eje de rotación (tornamesa), sin oclusión ni desenfoque por movimiento. En planta habrá: cinta con
textura y reflejos, iluminación variable, oclusión parcial/objetos múltiples, movimiento de la cinta (blur),
pose de cámara y escala variables, empaques nuevos. Mitigaciones: (1) aumentación de datos (brillo/contraste,
blur, ruido, escala, recortes) y captura de vistas reales por SKU en la estación; (2) compuerta de calidad
(oscuridad, saturación, nitidez, cuadro vacío) antes del modelo; (3) re-calibrar umbrales con datos de planta
(los umbrales son configurables en caliente vía `/v1/admin/thresholds`); (4) **piloto en sombra** con revisión
humana del 100% antes de habilitar ACT, y monitoreo de la tasa ACT/REVIEW/REJECT y de la deriva de distancias.

## 3. Protocolos y esquema de validación

| Protocolo | Train (por objeto) | Test | Validación (solo dentro de train) |
|---|---|---|---|
| A Interpolación | 0,10,…,350 (36) | 5,15,…,355 (36) | fit paso 20°, val el resto (brecha 10° > 5° del test) |
| B Sector ciego | 60 vistas | arco contiguo de 60° (12), inicio aleatorio por objeto (semilla 42) | se oculta otro arco de 60° dentro de train (semilla 43) |
| C Escasez | 1 cada 30° (12) | las otras 60 | fit paso 60°, val el resto |
| D estricto | 90 objetos, **todas** sus vistas (72) | solo las 720 vistas de los 10 objetos desconocidos | 10 pseudo-desconocidos entre los 90 (semilla 43); fit paso 10° de los otros 80 |
| D variante | 90 objetos, paso 10° (36) | vistas impares de los 90 + todas las de los 10 desconocidos | 10 pseudo-desconocidos entre los 90 (semilla 43) |

**D estricto** es el protocolo pedido: como el test solo tiene desconocidos, no se puede calcular AUROC; se
reporta la tasa de desconocidos rechazados y la de **desconocidos que llegan a ACT** (falsa aceptación). La
**variante D** (experimento adicional) entrena con la mitad de las vistas de los conocidos para poder medir
*ambos lados* con vistas no vistas (AUROC, conocidos rechazados por error). La validación imita (y suele exagerar)
la brecha de cada protocolo; en C la brecha de validación (30°) es mayor que la del test (≤15°).

In [6]:
rows = []
for p in ("A", "B", "C", "D_strict", "D"):
    sp = protocol_split(ds.labels, ds.angles, p); vs = validation_split(ds.labels, ds.angles, p)
    rows.append({"protocolo": p, "train": len(sp["train"]), "test": len(sp["test"]), "fit (val)": len(vs["fit"]),
                 "val": len(vs["val"]), "fuga train∩test": len(np.intersect1d(sp["train"], sp["test"])),
                 "val ⊂ train": bool(np.isin(np.r_[vs["fit"], vs["val"]], sp["train"]).all())})
display(pd.DataFrame(rows))
spB = protocol_split(ds.labels, ds.angles, "B"); spD = protocol_split(ds.labels, ds.angles, "D")
print("B: inicio del arco ciego (primeros 10 objetos):", {f"obj{k}": v for k, v in list(spB["arc_starts"].items())[:10]})
print("D: objetos desconocidos (semilla 42):", [f"obj{o}" for o in spD["unknown_objects"]])

,protocolo,train,test,fit (val),val,fuga train∩test,val ⊂ train
0,A,3600,3600,1800,1800,0,True
1,B,6000,1200,4800,1200,0,True
2,C,1200,6000,600,600,0,True
3,D_strict,6480,720,2880,3600,0,True
4,D,3240,3960,1440,1800,0,True


B: inicio del arco ciego (primeros 10 objetos): {'obj1': 30, 'obj2': 275, 'obj3': 235, 'obj4': 155, 'obj5': 155, 'obj6': 305, 'obj7': 30, 'obj8': 250, 'obj9': 70, 'obj10': 30}
D: objetos desconocidos (semilla 42): ['obj9', 'obj10', 'obj20', 'obj42', 'obj61', 'obj69', 'obj72', 'obj83', 'obj95', 'obj97']


## 4. Exploración: comparación de clasificadores y selección de la receta

**Características clásicas** (`features.py`, ~3 ms/imagen): histograma HSV 16×4×4 sobre la máscara de objeto
(raíz cuadrada → Hellinger), HOG 64×64 (9 orientaciones, celdas 8×8), LBP uniforme en rejilla 2×2 y momentos
de Hu; estandarización + PCA(150) ajustados **solo con train**.

**Clasificadores comparados** (rejilla pequeña, elegida en validación): kNN (k∈{1,5}), regresión logística
(C∈{0.1,1,10}), SVM RBF con probabilidades de Platt (C∈{1,10}) y Random Forest (300 árboles).

**Ángulo:** recuperación condicionada al objeto — entre las vistas de referencia de la clase predicha se toma
la más cercana y se refina con la media circular ponderada de los k vecinos cercanos a ese modo (k∈{1,2,3} se
elige en validación). La **incertidumbre angular** es un *score heurístico expresado en grados*: la distancia del
query a su vecino más cercano se traduce, con la curva distancia-vs-separación del propio objeto, en "cuántos
grados de giro equivalen a esa diferencia de apariencia". **No es una cota ni un intervalo del error real**; su
umbral se elige por el criterio empírico de riesgo (sección 5.3).

**Decisión:** distancia al vecino más cercano (novedad) → REJECT; confianza de clase baja, incertidumbre alta o
ambigüedad rotacional → REVIEW. Umbrales en validación: máxima fracción automatizada con error entre ACT ≤ 2 %
y **al menos 50 decisiones ACT en validación**; si no se cumple, la automatización se desactiva (todo a REVIEW)
y el riesgo se reporta N/A. Los riesgos se acompañan de intervalos de Wilson 95 %.

**Comparaciones obligatorias:** línea base 1-NN en píxeles crudos 32×32 (ángulo = el del vecino, sin rechazo:
siempre actúa) y referencia moderna: **ResNet18 ImageNet congelada** (embedding 512-d) con el mismo pipeline.

In [7]:
OUT_DIR = ROOT / "data" / "runtime" / "experiments"
t = time.time()
results = vexp.run_all(data, features=("classic", "resnet18"), out_dir=OUT_DIR, log=lambda *a: None)
print(f"exploración: {time.time() - t:.0f}s")
vrows = []
for p, rr in results["protocols"].items():
    for k, m in rr["classic"]["validation"].items():
        clf, params = k.split(":", 1)
        vrows.append({"protocolo": p, "clasificador": clf, "params": params, "val_exactitud": m["class_accuracy"]})
vdf = pd.DataFrame(vrows)
display(vdf.pivot_table(index=["clasificador", "params"], columns="protocolo", values="val_exactitud").round(4))
prod = vexp.select_production_classifier(results, "classic")
print("Mejor exactitud media de VALIDACIÓN en A–D:", prod["clf"], prod["params"])
display(pd.DataFrame({k: v for k, v in prod["mean_val"].items() if not k.startswith("_")}).T.round(4))

2026/10/24 12:48:03 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/nico/Documents/IA/.venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


exploración: 288s


protocolo                                 A       B       C       D
clasificador  params                                               
knn           {"n_neighbors": 1}     0.9772  0.8817  0.7817  0.9757
              {"n_neighbors": 5}     0.9683  0.8433  0.7467  0.9688
logreg        {"C": 0.1}             0.9933  0.9517  0.8783  0.9917
              {"C": 1.0}             0.9917  0.9542  0.8817  0.9917
              {"C": 10.0}            0.9917  0.9467  0.8800  0.9917
random_forest {"n_estimators": 300}  0.9689  0.8842  0.8050  0.9681
svm_rbf       {"C": 1.0}             0.9761  0.9292  0.7650  0.9701
              {"C": 10.0}            0.9928  0.9342  0.7850  0.9903

Mejor exactitud media de VALIDACIÓN en A–D: logreg {'C': 1.0}


,mean_val_class_accuracy,mean_val_end_to_end
knn,0.9041,0.4999
logreg,0.9552,0.5061
svm_rbf,0.9256,0.5058
random_forest,0.9065,0.4948


La regresión logística tiene la mejor exactitud media de validación (≈3 puntos sobre SVM RBF y ≈5 sobre kNN y
Random Forest) y además es la mejor calibrada (tabla de ECE abajo), rápida de entrenar y con artefacto pequeño. Con
esto se **congela la receta**: características clásicas + PCA(150) + logística C=1 + el procedimiento de umbrales.
La tabla siguiente es el test de *cada* clasificador de la exploración (cada uno con su mejor configuración de
validación por protocolo): es informativa y no se usa para elegir.

In [8]:
erows = []
for p, rr in results["protocols"].items():
    for feat in ("classic", "resnet18"):
        r = rr[feat]
        for clf, m in r["test"].items():
            erows.append({"protocolo": p, "rep": feat, "clf": clf, "exactitud": m["class_accuracy"], "MAE°": m["angle_mae_deg"],
                          "éxito e2e": m["end_to_end_success"], "ECE": m["ece"]})
edf = pd.DataFrame(erows)
display(edf.pivot_table(index=["rep", "clf"], columns="protocolo", values="exactitud").round(4))
display(edf.pivot_table(index=["rep", "clf"], columns="protocolo", values="ECE").round(4))

protocolo                    A       B       C       D
rep      clf                                          
classic  knn            0.9978  0.9167  0.9778  0.9972
         logreg         0.9997  0.9700  0.9912  1.0000
         random_forest  0.9986  0.9300  0.9685  0.9972
         svm_rbf        1.0000  0.9617  0.9678  1.0000
resnet18 knn            1.0000  0.9900  0.9977  1.0000
         logreg         1.0000  0.9967  0.9995  1.0000
         random_forest  0.9997  0.9808  0.9997  1.0000
         svm_rbf        1.0000  0.9967  0.9988  1.0000

protocolo                    A       B       C       D
rep      clf                                          
classic  knn            0.0022  0.0833  0.0222  0.0028
         logreg         0.0143  0.0177  0.0181  0.0005
         random_forest  0.3547  0.4644  0.5149  0.3522
         svm_rbf        0.5385  0.4987  0.7359  0.5109
resnet18 knn            0.0000  0.0100  0.0287  0.0000
         logreg         0.0035  0.0154  0.0523  0.0187
         random_forest  0.1931  0.2850  0.3717  0.1853
         svm_rbf        0.6570  0.5640  0.8408  0.6300

**Calibración.** La logística tiene ECE bajo en todos los protocolos (su confianza ≈ probabilidad real de
acierto); SVM con Platt multiclase y Random Forest están mal calibrados aunque acierten parecido, por lo que sus
probabilidades crudas no deben usarse para decidir.

## 5. Receta productiva congelada: evaluación independiente por protocolo (cifras de cabecera)

Receta (`src/vision/recipe.py`): características clásicas → PCA(150) → logística C=1; k angular ∈ {1,2,3} elegido
en validación; clases con ángulo poco fiable en validación → `rotational_ambiguity`; umbrales con el procedimiento
de la sección 4. En cada protocolo se ajusta **solo** con su train (validación dentro de train) y se evalúa **una
vez** en su test. La misma receta se aplica con embeddings ResNet18 como referencia moderna.

In [9]:
from src.vision import recipe as vrec
t = time.time()
R = vrec.run_recipe_all(data, features=("classic", "resnet18"), out_dir=OUT_DIR)
print(f"receta: {time.time() - t:.0f}s")
RS = va.build_recipe_summary(R["results"], R["baselines"])
def fmt_ci(x):
    return "N/A" if x is None or x[0] is None else f"[{x[0]:.2%}, {x[1]:.2%}]"
hrows = []
for p, row in RS.items():
    for key, lab in (("recipe_classic", "Receta clásica (prod.)"), ("baseline_raw_1nn", "Base píxeles 1-NN (siempre ACT)"),
                     ("recipe_resnet18", "Receta con ResNet18")):
        m = row.get(key)
        if not m: continue
        hrows.append({"protocolo": p, "modelo": lab, "k": m.get("k"), "exactitud": m["class_accuracy"], "MAE°": m["angle_mae_deg"],
                      "≤10°": m["angle_within_10"], "éxito e2e": m["end_to_end_success"], "% ACT": m["coverage_act"],
                      "n ACT": m.get("n_act"), "error|ACT": "N/A" if m["error_among_act"] is None else round(m["error_among_act"], 4),
                      "IC95 error|ACT": fmt_ci(m.get("error_among_act_wilson95")),
                      "automatización": m.get("automation_enabled", "—")})
hdf = pd.DataFrame(hrows); display(hdf)
thr = pd.DataFrame({p: {**R["results"]["classic"][p]["thresholds"], "k": R["results"]["classic"][p]["k_selected_on_val"],
                        "val n ACT": R["results"]["classic"][p]["threshold_validation"]["val_n_act"],
                        "val error|ACT": R["results"]["classic"][p]["threshold_validation"]["val_error_among_act"]}
                    for p in RS}).T
print("Umbrales de la receta clásica en cada protocolo (todos elegidos en validación):"); display(thr)
fig = va.recipe_bars(RS); va.savefig(fig, "03_per_protocol"); plt.show()

receta: 24s


,protocolo,modelo,k,exactitud,MAE°,≤10°,éxito e2e,% ACT,n ACT,error|ACT,IC95 error|ACT,automatización
0,A,Receta clásica (prod.),2.0,1.000000,2.706623,0.980833,0.980833,0.921667,3318.0,0.0021,"[0.10%, 0.43%]",True
1,A,Base píxeles 1-NN (siempre ACT),NaN,0.999722,5.928036,0.976104,0.975833,1.000000,NaN,0.0242,"[1.96%, 2.97%]",—
2,A,Receta con ResNet18,2.0,1.000000,1.982566,0.987778,0.987778,0.000000,0.0,N/A,N/A,False
3,B,Receta clásica (prod.),1.0,0.970000,66.469072,0.286082,0.277500,0.000000,0.0,N/A,N/A,False
4,B,Base píxeles 1-NN (siempre ACT),NaN,0.929167,61.224215,0.307623,0.285833,1.000000,NaN,0.7142,"[68.80%, 73.90%]",—
5,B,Receta con ResNet18,1.0,0.996667,48.323579,0.280100,0.279167,0.000000,0.0,N/A,N/A,False
6,C,Receta clásica (prod.),1.0,0.991167,13.127627,0.704725,0.698500,0.000000,0.0,N/A,N/A,False
7,C,Base píxeles 1-NN (siempre ACT),NaN,0.973833,14.024474,0.762793,0.742833,1.000000,NaN,0.2572,"[24.63%, 26.84%]",—
8,C,Receta con ResNet18,1.0,0.999833,11.351059,0.698283,0.698167,0.000000,0.0,N/A,N/A,False
9,D,Receta clásica (prod.),3.0,1.000000,3.074654,0.984259,0.984259,0.770455,3051.0,0.0066,"[0.42%, 1.01%]",True


Umbrales de la receta clásica en cada protocolo (todos elegidos en validación):


,min_class_confidence,max_angle_uncertainty_deg,unknown_distance,k,val n ACT,val error|ACT
A,0.40,30.0,36.291108,2.0,1530.0,0.015033
B,0.60,-1.0,38.781917,1.0,0.0,NaN
C,0.70,-1.0,42.352093,1.0,0.0,NaN
D,0.90,12.5,24.485617,3.0,667.0,0.014993
D_strict,0.95,30.0,22.463532,1.0,2696.0,0.002967


`error|ACT` = P(clase incorrecta, objeto desconocido aceptado o error angular > 10° | el robot actúa), con IC de
Wilson 95 %. *éxito e2e* = clase correcta **y** error circular ≤ 10° sobre las vistas conocidas, sin rechazo.
La línea base siempre actúa: en D sus desconocidos cuentan como error entre ACT (en D estricto su error es 100 %).
"N/A" = la receta no autorizó ninguna acción (automatización desactivada por falta de evidencia en validación).

### 5.1 Ángulo: error circular y elección de k en validación

In [10]:
outs = {p: va.load_outputs("recipe_classic", p, OUT_DIR) for p in ("A", "B", "C", "D")}
fig = va.angle_error_hist({p: outs[p] for p in "ABC"}); va.savefig(fig, "04_angle_error_hist"); plt.show()
kt = pd.DataFrame([{"protocolo": p, **r} for p in ("A", "B", "C", "D", "D_strict") for r in R["results"]["classic"][p]["k_table_val"]])
display(kt.pivot_table(index="k", columns="protocolo", values=["val_within_10", "val_mae_deg"]).round(3))
print("k elegido en validación:", {p: R["results"]["classic"][p]["k_selected_on_val"] for p in RS})

val_mae_deg                                  val_within_10                            
protocolo           A       B       C       D D_strict             A      B    C      D D_strict
k                                                                                               
1              15.042  61.581  52.798  15.924    6.323         0.807  0.280  0.0  0.803    0.983
2               8.585  62.449  52.798   9.632    2.967         0.942  0.168  0.0  0.936    0.983
3               9.538  63.251  52.798  10.562    3.430         0.941  0.160  0.0  0.936    0.982

k elegido en validación: {'A': 2, 'B': 1, 'C': 1, 'D': 3, 'D_strict': 1}


**Lectura.** La métrica es el error circular `min(|a−b|, 360−|a−b|)`. La tabla de k es de **validación** (no de
test): interpolar con 2–3 vecinos ayuda cuando el query cae entre dos vistas de referencia cercanas (A, D), pero
no cuando la brecha es grande (B, C), donde k=1 queda igual o mejor — por eso k se elige por protocolo en
validación. En **B** (arco ciego de 60°) ningún método basado en apariencia puede "inventar" las vistas no vistas;
en **C** (12 vistas) el ángulo es útil pero no alcanza la confiabilidad exigida para automatizar.

### 5.2 Confianza: calibración, riesgo–cobertura y desconocidos

In [11]:
fig = va.reliability_and_risk(outs); va.savefig(fig, "05_reliability_risk"); plt.show()
fig = va.unknown_roc(outs["D"]); va.savefig(fig, "06_unknown_roc"); plt.show()
for p in ("D", "D_strict"):
    d = R["results"]["classic"][p]["decision"]
    print(p, {k: (round(v["rate"], 4), [round(x, 4) for x in v["wilson95"]]) for k, v in d.items() if k in ("unknown_rejected", "unknown_reaches_act")},
          "| AUROC combinado:", round(R["results"]["classic"][p].get("auroc_combined_rank", float("nan")), 4),
          "| conocidos rechazados:", round(d.get("known_false_reject", float("nan")), 4))

D {'unknown_rejected': (np.float64(0.7125), [0.6784, 0.7444]), 'unknown_reaches_act': (np.float64(0.0153), [0.0086, 0.0271])} | AUROC combinado: 0.9926 | conocidos rechazados: 0.0151
D_strict {'unknown_rejected': (np.float64(0.775), [0.7431, 0.804]), 'unknown_reaches_act': (np.float64(0.0486), [0.0352, 0.0669])} | AUROC combinado: nan | conocidos rechazados: nan


**¿Es la confianza una base válida para decisiones automáticas?** (1) La logística está bien calibrada (diagrama
de confiabilidad, ECE bajo). (2) En las curvas riesgo–cobertura, priorizar los casos de mayor confianza o menor
incertidumbre deja el riesgo de los primeros tramos de cobertura muy por debajo del riesgo global; las curvas
**no** son estrictamente monótonas (hay subidas locales con pocos casos) y en B no existe un tramo con riesgo
aceptable. (3) En D (variante) distancia y confianza separan conocidos de desconocidos con AUROC alto; en D
estricto se reporta qué fracción de desconocidos se rechaza y cuántos llegan a ACT. Los desconocidos que llegan a
ACT son casi siempre "casi duplicados" de un conocido (p. ej. obj61 ≈ obj92): la confianza no los separa.

### 5.3 Umbral de incertidumbre vs. tolerancia ±10°

El umbral de incertidumbre (p. ej. 30° en producción) **no** significa "error ≤ 30°": es un corte sobre un score
heurístico. Lo que importa es el riesgo empírico `P(clase incorrecta o |error| > 10° | ACT)`. La figura lo
muestra en función del umbral, en **validación** (donde se elige: el mayor umbral cuyo riesgo ≤ 2 % con ≥ 50 ACT)
y en el **test** de la receta (solo descriptivo), con IC de Wilson 95 %. Los umbrales altos son seguros porque
la ambigüedad y la confianza ya filtran los casos peligrosos; el score angular solo debe cortar la cola.

In [12]:
fig = va.sweep_figure(R["results"]["classic"], ("A", "D")); va.savefig(fig, "09_uncertainty_sweep"); plt.show()
sw = pd.DataFrame([{"protocolo": p, **{k: v for k, v in r.items() if k in ("max_angle_uncertainty_deg", "coverage", "risk_rate", "risk_n")}}
                   for p in ("A", "D") for r in R["results"]["classic"][p]["test_uncertainty_sweep_descriptive"]])
display(sw.round(4))

,protocolo,max_angle_uncertainty_deg,coverage,risk_rate,risk_n
0,A,2.5,0.0008,0.0000,3
1,A,5.0,0.1858,0.0015,669
2,A,7.5,0.7583,0.0011,2730
3,A,10.0,0.9125,0.0021,3285
4,A,12.5,0.9183,0.0021,3306
5,A,15.0,0.9203,0.0021,3313
6,A,20.0,0.9214,0.0021,3317
7,A,30.0,0.9217,0.0021,3318
8,A,45.0,0.9219,0.0021,3319
9,A,60.0,0.9219,0.0021,3319


## 6. Análisis de errores (receta clásica)

In [13]:
fails = va.per_object_failures({p: outs[p] for p in "ABC"}, top=8)
for p, rows in fails.items():
    print(f"Protocolo {p}: objetos con más fallos e2e"); display(pd.DataFrame(rows))
for p in ("B", "C"):
    fig = va.failure_examples_figure(ds, outs, p); va.savefig(fig, f"07_failures_{p}"); plt.show()
fA = outs["A"]; okA = (fA["label"] == fA["true_label"]) & (circular_error(fA["angle"], fA["true_angle"]) <= 10)
actA = fA["decision"] == "ACT"
print("A: fallos e2e totales:", int((~okA).sum()), "| fallos que llegaron a ACT:", int((~okA & actA).sum()))

Protocolo A: objetos con más fallos e2e


,object,failures,n,class_errors,confused_with
0,obj95,17,36,0,None
1,obj47,14,36,0,None
2,obj94,12,36,0,None
3,obj98,7,36,0,None
4,obj67,5,36,0,None
5,obj35,3,36,0,None
6,obj84,3,36,0,None
7,obj13,1,36,0,None


Protocolo B: objetos con más fallos e2e


,object,failures,n,class_errors,confused_with
0,obj98,12,12,0,None
1,obj84,12,12,0,None
2,obj95,11,12,0,None
3,obj79,11,12,0,None
4,obj67,11,12,0,None
5,obj1,10,12,0,None
6,obj69,10,12,0,None
7,obj72,10,12,0,None


Protocolo C: objetos con más fallos e2e


,object,failures,n,class_errors,confused_with
0,obj47,34,60,0,NaN
1,obj95,32,60,0,NaN
2,obj94,31,60,0,NaN
3,obj35,28,60,0,NaN
4,obj84,27,60,10,obj59
5,obj67,25,60,7,obj37
6,obj49,23,60,0,NaN
7,obj64,23,60,0,NaN


A: fallos e2e totales: 69 | fallos que llegaron a ACT: 7


**Lectura.** Los fallos angulares se concentran en pocos objetos: los de mayor índice de perfil (donas y taza:
obj95, obj94, obj47) y los de mayor índice LOO (vistas "gemelas" a 180°: obj98, obj67, obj84). El error en esos
casos es de ±180° o "cualquier ángulo", no de unos pocos grados, y la mayoría se envía a REVIEW por
`rotational_ambiguity` o `high_angle_uncertainty`. Los errores de clase se dan entre objetos de la misma familia
visual (pares casi duplicados de la sección 2). En B el fallo dominante es estructural (arco no visto).

## 7. Costo: latencia y tamaño del artefacto (clásico vs. referencia moderna)

In [14]:
import subprocess
r = subprocess.run([sys.executable, str(ROOT / "scripts" / "train_vision.py")], capture_output=True, text=True, cwd=ROOT)
print(r.stdout[-1500:]); print(r.stderr[-800:] if r.returncode else "")
from src.vision.inference import VisionPredictor
pred = VisionPredictor.load(ROOT / "models" / "vision-1.0.0")
rng = np.random.default_rng(42); sample = ds.images[rng.choice(len(ds.images), 100, replace=False)]
lat = {"classic (pipeline completo, 1 imagen)": va.measure_latency(pred, sample)}
lat["baseline píxeles 1-NN (media/imagen en lote)"] = {"mean_ms": R["baselines"]["A"]["mean_ms_per_image_in_batch"]}
rl = va.measure_resnet_latency(sample, n=50)
lat["resnet18 CPU (solo embedding, 1 imagen)"] = rl["cpu"]
if "mps" in rl: lat["resnet18 MPS (solo embedding, 1 imagen)"] = rl["mps"]
display(pd.DataFrame(lat).T.round(2))
size_mb = sum(f.stat().st_size for f in (ROOT / "models" / "vision-1.0.0").glob("*") if f.is_file()) / 1e6
print(f"Artefacto clásico: {size_mb:.1f} MB | pesos ResNet18 fp32: {rl['weights_mb_fp32']} MB (+ torch/torchvision)")
fig = va.latency_figure(lat); va.savefig(fig, "08_latency"); plt.show()
COST = {"latency": lat, "classic_artifact_mb": size_mb, "resnet18_weights_mb": rl["weights_mb_fp32"]}

Receta congelada: classic-logreg-v1 (logreg {'C': 1.0}, PCA 150)
k (validación): 1 | clases con ángulo poco fiable: ['obj47', 'obj94', 'obj95']
Umbrales (validación): {'min_class_confidence': 0.99, 'max_angle_uncertainty_deg': 30.0, 'unknown_distance': 19.47063943862915} {'val_n_act': 2901, 'val_n_errors': 21, 'val_coverage': 0.7325757575757575, 'val_error_among_act': 0.007238883143743537, 'val_error_wilson95': [0.004739572855841491, 0.011041528599053671], 'target_error': 0.02, 'min_act': 50, 'angle_threshold_source': 'validation', 'automation_enabled': True}
Entrenamiento final: 0.9s; clases simétricas: ['obj47', 'obj67', 'obj84', 'obj94', 'obj95', 'obj98']
Artefacto exportado en /Users/nico/Documents/IA/models/vision-1.0.0 (9.4 MB, sha256 19cbc8a4516b0ecb…)




,p50_ms,p95_ms,mean_ms
"classic (pipeline completo, 1 imagen)",13.32,18.41,14.03
baseline píxeles 1-NN (media/imagen en lote),NaN,NaN,0.27
"resnet18 CPU (solo embedding, 1 imagen)",30.46,35.01,NaN
"resnet18 MPS (solo embedding, 1 imagen)",15.69,19.87,NaN


Artefacto clásico: 9.4 MB | pesos ResNet18 fp32: 44.7 MB (+ torch/torchvision)


**¿Justifica la brecha el costo?** Con la misma receta, ResNet18 mejora la exactitud de clase en los protocolos
con brecha (B, C) y el MAE angular, pero no resuelve el arco ciego ni habilita automatizar C; en A/D la comparación
de fracción automatizada y error entre ACT está en la tabla de la sección 5. Solo el embedding de ResNet18 ya
cuesta más en CPU que el pipeline clásico completo, añade ~45 MB de pesos más la dependencia de PyTorch y es menos
interpretable. **Producción = receta clásica**; ResNet18 queda como referencia y candidata si el piloto en planta
muestra una brecha mayor (en particular en desconocidos casi duplicados).

## 8. Exportación del artefacto y registro

In [15]:
results["production_selection"] = {"clf": prod["clf"], "params": prod["params"], "criterion": "max exactitud media de validación A–D (exploración)"}
results["production_selection_scores"] = prod["mean_val"]
results["recipe"] = vexp.to_jsonable(R)
results["recipe_summary"] = vexp.to_jsonable(RS)
results["exploration_summary"] = va.build_summary(results)
results["cost"] = COST
results["symmetry"] = {"profile_score": {f"obj{k}": v for k, v in sym["profile_score"].items()},
                       "loo_score": {f"obj{k}": v for k, v in sym["loo_score"].items()}}
results["confusable_pairs"] = pairs
results["error_analysis"] = fails
p = va.write_results(vexp.to_jsonable(results)); print("escrito", p)
# re-exporta para que metadata.json lleve el resumen de la receta
r = subprocess.run([sys.executable, str(ROOT / "scripts" / "train_vision.py")], capture_output=True, text=True, cwd=ROOT)
print(r.stdout[-1200:])
meta = json.loads((ROOT / "models" / "vision-1.0.0" / "metadata.json").read_text())
print({k: meta[k] for k in ("version", "classifier", "k_angle_refine", "thresholds", "symmetric_classes", "artifact_size_mb", "model_sha256", "git")})
print(f"Tiempo total del notebook: {(time.time() - T0) / 60:.1f} min")

escrito /Users/nico/Documents/IA/docs/report/results.json


Receta congelada: classic-logreg-v1 (logreg {'C': 1.0}, PCA 150)
k (validación): 1 | clases con ángulo poco fiable: ['obj47', 'obj94', 'obj95']
Umbrales (validación): {'min_class_confidence': 0.99, 'max_angle_uncertainty_deg': 30.0, 'unknown_distance': 19.47063943862915} {'val_n_act': 2901, 'val_n_errors': 21, 'val_coverage': 0.7325757575757575, 'val_error_among_act': 0.007238883143743537, 'val_error_wilson95': [0.004739572855841491, 0.011041528599053671], 'target_error': 0.02, 'min_act': 50, 'angle_threshold_source': 'validation', 'automation_enabled': True}
Entrenamiento final: 1.1s; clases simétricas: ['obj47', 'obj67', 'obj84', 'obj94', 'obj95', 'obj98']
Artefacto exportado en /Users/nico/Documents/IA/models/vision-1.0.0 (9.4 MB, sha256 19cbc8a4516b0ecb…)

{'version': 'vision-1.0.0', 'classifier': {'name': 'logreg', 'params': {'C': 1.0}, 'selection_source': 'receta congelada (src/vision/recipe.py), elegida en la exploración por validación'}, 'k_angle_refine': 1, 'thresholds': {'min

## 9. Conclusiones

* **Receta productiva:** características clásicas (HSV + HOG + LBP + Hu) → PCA(150) → regresión logística C=1;
  ángulo por recuperación condicionada al objeto con un score de incertidumbre en grados; decisión
  ACT/REVIEW/REJECT con umbrales fijados en validación. El artefacto `vision-1.0.0` es esta receta ajustada con
  las 7.200 vistas: sus métricas esperadas son las de la receta en los protocolos (sección 5), no un test propio.
* **Protocolos:** A y D (variante) son automatizables con error entre ACT bajo (ver IC de Wilson en la tabla 5);
  B (arco ciego) y C (12 vistas/objeto) **no** lo son: la validación no aporta evidencia suficiente y la receta
  desactiva la automatización (todo a REVIEW). En D estricto una fracción pequeña de desconocidos llega a ACT
  (casi duplicados). Para dar de alta un SKU: **≥ 12 vistas cubriendo ≥ 300°** para staging, y referencias densas
  (idealmente cada 5–10°) para habilitar ACT.
* **Limitaciones:** ver la model card (`docs/model_card.md`) — dataset de laboratorio, un solo eje de rotación,
  sin oclusión ni variación de fondo/iluminación; requiere piloto en sombra y recalibración en planta. Ninguna
  regla de decisión *garantiza* acierto: reduce el riesgo a la tasa medida.